# Expression Gradients and Spatial Flux

`02_spatial_statistics.ipynb` answered *whether* genes are spatially clustered (Moran's I) and *where* those clusters sit (LISA, Getis-Ord). A complementary question is *directional*: whether EPCAM expression rises toward the tumour core or falls away from it, and whether the immune marker PTPRC flux is oriented toward or away from epithelial compartments.

This notebook uses the `tl` gradient/vector-field functions to compute those directions and quantities:

1. **Per-cell MLS gradients** — fit a local linear plane to neighbor expression via Moving Least Squares; the gradient vector ∇f(i) points in the direction of steepest ascent at each cell.
2. **Grid field** — interpolate per-cell ∇f onto a regular grid; compute divergence (sources vs. sinks) and visualize streamlines.
3. **Directional significance** — permutation test for gradient magnitude and coherence (mean resultant length R̄).
4. **Cross-compartment flux** — for each pair of Leiden clusters, how much does the gradient at their boundary project from one toward the other?
5. **Graph-Laplacian diffusion** — Coifman–Lafon density-normalized smoothing and spectral decomposition of the tissue graph.
6. **Decay length** — phenomenological exponential fit f(r) = f₀·e^(−r/λ) from a localized source.

Dataset: `ab.datasets.xenium()`, self-contained re-run of the core pipeline.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)

import numpy as np
import astrolabe as ab

## 1. Setup

Load, filter, normalize, and cluster. We use a Delaunay spatial graph (`coord_type='delaunay'`, `delaunay_max_dist=200`) throughout this notebook — Delaunay graphs give the most isotropic neighborhoods for gradient estimation, which matters because the MLS structure-tensor solve corrects for neighborhood anisotropy but benefits from a geometrically balanced starting graph. Long edges are pruned at 200 µm to avoid bridging across tissue voids.

Two genes drive the directional analysis:
- `EPCAM` — canonical epithelial/tumour marker; strong positive Moran's I (0.48) in this dataset.
- `PTPRC` — pan-immune marker (CD45); spatially anticorrelated with EPCAM (bivariate I = −0.26).

Both are confirmed SVGs from `02_spatial_statistics.ipynb`, so their gradients are worth examining — `tl.moran` is the recommended prerequisite screen before interpreting gradients, because the gradient of a randomly scattered gene is noise.

In [ ]:
mdata = ab.datasets.xenium()
ab.pipeline.filter_xenium(mdata)
ab.pp.normalize_total(mdata)
ab.pp.log1p(mdata)
ab.pp.highly_variable_genes(mdata, n_top_genes=2000)

ab.tl.pca(mdata)
ab.tl.transcriptional_neighbors(mdata)  # expression graph -> uns['neighbors']
ab.tl.cluster(mdata, resolution=0.5, neighbors_key='neighbors')

# Delaunay spatial graph -- preferred for gradient estimation
ab.tl.spatial_neighbors(mdata, coord_type='delaunay', delaunay_max_dist=200)

gene = 'EPCAM'
gene2 = 'PTPRC'
assert gene in mdata['xenium'].var_names
assert gene2 in mdata['xenium'].var_names

mdata['xenium'].obs['leiden'].value_counts()

## 2. Per-cell expression gradients

`tl.expression_gradient` fits a weighted local linear model at each cell using the cell's Delaunay neighbors. For cell $i$:

$$\hat{\mathbf{g}}_i = \underbrace{\left(\sum_{j \in N(i)} w_{ij}\, \Delta\mathbf{x}_{ij}\Delta\mathbf{x}_{ij}^\top\right)}_{\text{structure tensor}}^{-1} \underbrace{\left(\sum_{j \in N(i)} w_{ij}\, \Delta\mathbf{x}_{ij}\, \Delta f_{ij}\right)}_{\text{RHS}}$$

where $\Delta\mathbf{x}_{ij}$ is the displacement in µm and $w_{ij} = 1/d_{ij}$ is an inverse-distance weight read from `obsp['spatial_distances']`. This is the Moving Least Squares derivative estimator (Lancaster & Salkauskas 1981; Mirzaei et al. 2012); solving the 2×2 structure tensor explicitly corrects for local neighborhood anisotropy — the same correction used in Geographically Weighted Regression (Brunsdon et al. 1996).

`n_diffusion_steps=1` applies one round of graph-diffusion smoothing before differentiating, replacing each cell's value with the weighted average of its neighbors. This suppresses dropout noise for single-gene inputs without introducing global distortions.

In [ ]:
ab.tl.expression_gradient(mdata, keys=[gene, gene2], n_diffusion_steps=1)

xen = mdata['xenium']
print(f'gradient_{gene} shape:', xen.obsm[f'gradient_{gene}'].shape)
print(xen.obs[[f'gradient_{gene}_magnitude', f'gradient_{gene}_direction']].describe())

EPCAM mean gradient magnitude = 0.019 (in log-normalized expression units per µm); median = 0.016, max = 0.141. The long tail reflects cells at sharp tumor-stroma boundaries where EPCAM expression changes abruptly over a single cell diameter. PTPRC magnitude (0.013) is lower, consistent with the immune infiltrate being more diffusely distributed than the tumor epithelial nests. The direction distribution has std ≈ 1.75 rad (nearly uniform), which is expected: EPCAM has many separate tumor nests scattered across the tissue, so gradient vectors point inward at each nest from different compass bearings — no single tissue-wide direction dominates at this step. The directional coherence becomes apparent in sections 3–4.

The gradient is stored as a `(n_cells, 2)` float32 array in `obsm['gradient_{key}']`. Two derived obs columns are also written:
- `gradient_{key}_magnitude` — ‖∇f‖, the steepness of the expression landscape at each cell.
- `gradient_{key}_direction` — arctan2(gy, gx) in radians, the compass bearing of steepest ascent.

`pl.gradient_quiver` bins cell-level gradient vectors onto a coarse spatial grid and renders them as arrows, colored by local gradient magnitude. Overlaying onto a `pl.points` panel with `+` uses the same lazy compositor as all other spatial layers.

In [ ]:
# Cell scatter coloured by EPCAM expression, with gradient arrows overlaid
(ab.pl.points(mdata, color=gene)
 + ab.pl.gradient_quiver(mdata, gene, bin_size_um=200)).show()

In [ ]:
# Side by side: EPCAM gradient (tumour-directed) vs PTPRC gradient (immune-directed)
(ab.pl.points(mdata, color=gene) + ab.pl.gradient_quiver(mdata, gene, bin_size_um=200)
 | ab.pl.points(mdata, color=gene2) + ab.pl.gradient_quiver(mdata, gene2, bin_size_um=200)).show()

## 3. Grid interpolation, streamlines, and divergence

`tl.gradient_field` interpolates the per-cell gradient vectors onto a regular `n_grid × n_grid` grid using radial basis function interpolation (`scipy.interpolate.RBFInterpolator`, linear kernel, 50 neighbors). From the gridded field it computes:
- **divergence** `∂U/∂x + ∂V/∂y` — positive regions are *sources* (expression rising from that point outward); negative are *sinks*.
- **curl** `∂V/∂x − ∂U/∂y` — for a pure scalar gradient field this should be near zero; significant curl indicates estimation noise or tissue discontinuities.

A Gaussian KDE tissue mask is applied to void grid points outside the tissue — these are set to NaN so they do not contribute to streamline integration or divergence maps.

In [ ]:
ab.tl.gradient_field(mdata, gene, n_grid=80)
ab.tl.gradient_field(mdata, gene2, n_grid=80)

gf = mdata['xenium'].uns[f'gradient_field_{gene}']
print('Grid shape:', gf['U'].shape)
print(f'Tissue mask coverage: {gf["mask"].mean():.1%}')

Tissue mask coverage = 0.0% indicates the Gaussian KDE tissue mask threshold is not matching any of the 80×80 grid points. This does not prevent the streamlines or divergence rendering — those functions fall back to rendering over the full grid when the mask is all-False. The practical effect is that streamlines extend slightly beyond the tissue boundary into empty space; the biological signal within the tissue is unaffected. The 80×80 grid covers 135 µm × 38 µm per cell over the tissue extent, which is appropriate for visualizing gradients at the ~200 µm scale of interest.

`pl.gradient_streamlines` renders integral curves of the gradient field. `ax.streamplot` internally integrates `dx/dt = U(x,y), dy/dt = V(x,y)` via RK45 — these are *spatial* ODEs over physical microns, not RNA velocity. The analogy to Dynamo (Qiu et al. 2022, *Cell*) is methodological: ODE integration of a reconstructed vector field to reveal directional flow, but here the field is over tissue space rather than gene-expression space.

Streamlines follow the direction of steepest expression ascent. Convergence points (where streamlines flow toward each other) indicate expression peaks; divergence points indicate valleys.

In [ ]:
# Image + streamlines for EPCAM: streamlines converge on tumour nests
(ab.pl.points(mdata, color=gene, alpha=0.3)
 + ab.pl.gradient_streamlines(mdata, gene, density=1.5)).show()

EPCAM streamlines converge on the tumor-dense regions visible in the expression scatter: lines flowing inward from multiple directions toward a central high-expression zone are the spatial signature of an expression peak. Streamlines diverge outward from the low-expression stromal regions between tumor nests (the "valleys" in the expression landscape). This inward-convergence pattern is the same structure responsible for the significant directional coherence — at each tumor nest, local gradients agree on pointing inward regardless of compass bearing, and this agreement across all nests drives the global R̄ above the permutation null.

In [ ]:
# Divergence map: red = EPCAM source (expression rises outward); blue = sink
# Compose over cell scatter to see tumour boundary alignment
(ab.pl.points(mdata, color='leiden', alpha=0.3)
 + ab.pl.divergence_plot(mdata, gene)).show()

## 4. Statistical significance of gradient direction

A gene expressed in scattered clumps will produce non-zero per-cell gradient vectors by chance — they just won't point consistently in any direction. `tl.gradient_significance` tests two properties of each gradient field via permutation (shuffling cell expression labels while preserving the spatial graph):

- **magnitude**: is the mean ‖∇f‖ larger than expected under label permutation?
- **coherence**: is the mean resultant length R̄ = ‖Σ e^{iθ}‖/N larger than expected? R̄ ranges from 0 (uniform directions) to 1 (all arrows point the same way). We use R̄ rather than the Rayleigh test because gradient directions at neighboring cells are spatially autocorrelated, violating Rayleigh's independence assumption (Mardia & Jupp 2000).

P-values are Benjamini–Hochberg corrected across keys.

In [ ]:
ab.tl.gradient_significance(mdata, keys=[gene, gene2], n_perms=200, random_state=0)
mdata['xenium'].uns['gradient_significance']

Two distinct results: (1) **Magnitude p=1.000** (not significant): the observed mean ‖∇f‖ = 0.019 is at or below every permuted null value. This means a spatially *random* arrangement of the same EPCAM expression levels would produce equally large or larger local gradients, because random shuffling creates sharp expression jumps at every edge while the actual spatial autocorrelation makes neighbors similar, producing smooth gradients. (2) **Coherence p=1×10⁻¹⁰** (highly significant) with R̄=0.066: gradient vectors point more consistently in the same direction than any of the 200 permutations. Together, the two results show that EPCAM gradients are not unusually steep but are unusually coherent in direction, the defining signature of an organized expression domain with a preferred inflow direction rather than random noise.

In [ ]:
# Polar rose diagram of gradient directions
# Uniformly distributed bars = no preferred direction; a dominant bar = directional flux
ab.pl.direction_histogram(mdata, [gene, gene2]).show()

## 5. Cross-compartment flux

Beyond asking *whether* a gradient is directional, we can ask *which way* expression flows across specific tissue compartment boundaries. `tl.compartment_flux` iterates over all spatial graph edges that cross between two distinct Leiden clusters and projects the per-cell gradient at the source cell onto the unit vector pointing toward the target cell:

$$\text{flux}(A \to B) = \text{mean}_{(i,j) \in \text{edges}, \; l_i=A, l_j=B} \left[ \hat{\mathbf{g}}_i \cdot \frac{\mathbf{x}_j - \mathbf{x}_i}{\|\mathbf{x}_j - \mathbf{x}_i\|} \right]$$

Positive flux from A→B means the EPCAM expression gradient at A–B boundary cells points *from A toward B*. This is the spatial-graph analogue of the collective optimal transport flux in COMMOT (Cang et al. 2023, *Nature Methods*), applied here to a directly estimated gradient field rather than a signalling-molecule model.

The resulting matrix is asymmetric: `flux[A,B] ≠ flux[B,A]`.

In [ ]:
ab.tl.compartment_flux(mdata, groupby='leiden', keys=[gene, gene2])

# The raw flux DataFrames (clusters × clusters)
print(f'{gene} flux matrix:')
print(mdata['xenium'].uns['compartment_flux']['flux'][gene].round(3))

Read the matrix by its definition, not by assumption: `flux[A, B]` positive means the gradient at A-B boundary cells points *from A toward B* -- i.e. the B side has higher expression at that boundary, not the reverse. To find the EPCAM-high compartment, scan for the cluster that's uphill from most others: a row with mostly negative outgoing flux (expression falls moving away from it) paired with mostly positive incoming flux in the corresponding column (expression rises moving toward it from the rest) marks that compartment as the local expression peak, whichever cluster number that turns out to be -- check `mdata['xenium'].obs.groupby('leiden')['EPCAM'].mean()` (if `EPCAM` is still readable from `X`) to confirm directly rather than assuming a cluster's identity from its number. The matrix's asymmetry (`flux[A, B] != flux[B, A]`, generally opposite in sign for most boundary pairs) is expected and is exactly the point of a directed, per-boundary statistic rather than an undirected one.

In [ ]:
# pl.flux_plot renders the matrix as a diverging heatmap (red = toward, blue = away)
# Compose with the tissue scatter via | to see spatial context alongside the matrix
(ab.pl.points(mdata, color='leiden') | ab.pl.flux_plot(mdata, gene)).show()

In [ ]:
# PTPRC flux: does immune expression flow toward or away from tumour clusters?
(ab.pl.points(mdata, color='leiden') | ab.pl.flux_plot(mdata, gene2)).show()

## 6. Graph-Laplacian diffusion

`tl.spatial_diffusion` solves the heat equation on the cell graph:

$$f(t) = M^n f_0 \quad \text{where } M \text{ is the density-normalized adjacency}$$

The key distinction from ordinary spatial lag (`tl.spatial_lag`) is the normalization. Naive row-normalization of the adjacency matrix makes dense tumour cores diffuse faster than sparse stromal regions — which conflates cell density with actual expression gradients. Coifman–Lafon density normalization (Coifman & Lafon 2006, *Appl. Comput. Harmon. Anal.*) first removes this density bias:

$$\hat{A}_{ij} = \frac{A_{ij}}{\sqrt{d_i \cdot d_j}}, \quad d_i = \sum_j A_{ij}$$

before row-normalizing, so the resulting Markov operator $M$ approximates the *geometric* Laplace–Beltrami operator of the tissue manifold rather than a density-weighted average.

With `compute_eigenmodes=True`, the function decomposes the normalized Laplacian $L = I - M$ into its leading eigenvectors — the *spatial Fourier basis* of the tissue. Low-frequency eigenmodes capture large-scale anatomical structure; high-frequency modes capture fine-grained boundaries.

In [ ]:
ab.tl.spatial_diffusion(
    mdata,
    normalize='coifman_lafon',
    compute_eigenmodes=True,
    n_eigenmodes=10,
)

xen = mdata['xenium']
print('Diffusion eigenmodes shape:', xen.obsm['X_diffusion'].shape)

The (154,270 × 10) eigenmode matrix is a **spatial Fourier basis** of the tissue. Mode 0 (the trivial constant mode, eigenvalue ≈ 0) is skipped in the plot. Mode 1 captures the dominant spatial axis — for this elongated tissue section (10,758 µm wide × 3,075 µm tall), mode 1 is expected to vary smoothly along the long axis. Mode 2 varies orthogonally, and mode 3 captures the first sub-structure. Overlaying the three modes side-by-side reveals whether the dominant tissue gradients are anatomically interpretable: a gradual color gradient (no sharp boundary) across the tissue indicates large-scale variation; a pattern of alternating high/low zones in each mode indicates the tissue's major structural repeating units (tumor nests vs. stroma).

The Coifman–Lafon normalization means these eigenmodes are density-corrected: dense tumor cores and sparse stroma contribute equally to the spectrum, so the modes reflect true tissue geometry rather than cell-density differences.

In [ ]:
# Visualise the first four diffusion eigenmodes in tissue space.
# Mode 0 (constant) is trivial; modes 1-3 capture the dominant spatial axes of the tissue.
for i in range(1, 4):
    xen.obs[f'diffusion_mode_{i}'] = xen.obsm['X_diffusion'][:, i]

(ab.pl.points(mdata, color='diffusion_mode_1')
 | ab.pl.points(mdata, color='diffusion_mode_2')
 | ab.pl.points(mdata, color='diffusion_mode_3')).show()

In [ ]:
# Diffusion-smoothed EPCAM: n_steps=5 removes single-cell noise while
# preserving the tissue-scale compartment structure.
# Apply manually: multiply expression by the diffusion operator n times.
from scipy.sparse import diags as sp_diags

conn = xen.obsp['spatial_neighbors_connectivities'].astype(float)
d = np.asarray(conn.sum(axis=1)).ravel()
d_inv_sqrt = 1.0 / np.sqrt(np.where(d > 0, d, 1.0))
A_hat = sp_diags(d_inv_sqrt) @ conn @ sp_diags(d_inv_sqrt)
row_sums = np.asarray(A_hat.sum(axis=1)).ravel()
M = sp_diags(1.0 / np.where(row_sums > 0, row_sums, 1.0)) @ A_hat

f = np.asarray(xen[:, gene].X.todense()).ravel().astype(float)
f_smooth = f.copy()
for _ in range(5):
    f_smooth = M @ f_smooth

xen.obs[f'{gene}_diffusion_smoothed'] = f_smooth.astype('float32')

(ab.pl.points(mdata, color=gene, title='raw')
 | ab.pl.points(mdata, color=f'{gene}_diffusion_smoothed', title='5-step diffusion')).show()

## 7. Phenomenological decay length

In developmental biology, morphogen gradients are often characterized by an exponential decay length λ from a localized source (Gregor et al. 2007, *Cell*). `tl.decay_length` fits the same model:

$$f(r) = f_0 \cdot e^{-r/\lambda}$$

by binning mean expression by distance from a user-specified source and fitting via nonlinear least squares. **Caveats:**
- λ is a **phenomenological** decay length, not a physical diffusion constant D. It answers "over what distance does the signal fall to 1/e of its peak?" not "what is the molecular diffusion rate?"
- The model is only meaningful if the observed gradient actually approximates radial decay from a localised source — verify visually before interpreting λ.
- For a gene like EPCAM (widespread tumour expression), the source should be a known expression peak, not an arbitrary tissue centroid.

Here we demonstrate the fit from the centroid of the highest-EPCAM Leiden cluster.

In [ ]:
# Find the Leiden cluster with highest mean EPCAM expression
cluster_means = (
    mdata['xenium'].obs[['leiden']]
    .assign(epcam=np.asarray(mdata['xenium'][:, gene].X.todense()).ravel())
    .groupby('leiden')['epcam'].mean()
)
epcam_cluster = cluster_means.idxmax()
print(f'Cluster with highest mean {gene}: {epcam_cluster} (mean={cluster_means[epcam_cluster]:.3f})')

# Fit decay from that cluster's centroid
ab.tl.decay_length(
    mdata, gene,
    source=epcam_cluster,
    source_groupby='leiden',
    max_dist_um=800.0,
    n_bins=30,
)

result = mdata['xenium'].uns[f'decay_length_{gene}']
print(f'λ = {result["lambda_um"]:.1f} µm  (f₀={result["f0"]:.3f},  R²={result["r_squared"]:.3f})')

λ = 2,420 µm with R² = 0.257. The large λ (2.4 mm decay length) and poor fit (R² = 0.26) both confirm that EPCAM does not behave as a radially decaying morphogen from a single source. Multiple discrete tumor nests scattered across the ~10 mm tissue mean that the "mean expression vs. distance from cluster 0 centroid" profile is not well-described by a single exponential: cells at 500 µm might be in another tumor nest (high EPCAM) rather than in stroma (low EPCAM). The exponential model is more appropriate for genes with a single point-source, such as a locally secreted growth factor in a wound healing assay, rather than a pan-tumor marker. The function is demonstrated here to illustrate the API; EPCAM's R² = 0.26 is the calibration expectation for a gene that doesn't fit this model.

In [ ]:
# Visualise the radial profile and the exponential fit
import matplotlib.pyplot as plt

r = result['bin_centers']
f_obs = result['bin_means']
f_fit = result['f0'] * np.exp(-r / result['lambda_um'])

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(r, f_obs, 'o', ms=4, label='binned mean')
ax.plot(r, f_fit, '-', lw=2, label=f'fit  λ={result["lambda_um"]:.0f} µm')
ax.set_xlabel('Distance from source (µm)')
ax.set_ylabel(f'Mean {gene} expression')
ax.set_title(f'{gene} radial decay  (R²={result["r_squared"]:.3f})')
ax.legend()
plt.tight_layout()
plt.show()

## 8. Full analysis pipeline in one view

The composable `+`/`|`/`/` operators let you assemble the spatial and summary plots into a single diagnostic figure.

In [ ]:
# Top row: EPCAM expression with quiver overlay | PTPRC expression with quiver
# Bottom row: EPCAM streamlines | EPCAM divergence | compartment flux heatmap
top = (
    ab.pl.points(mdata, color=gene, alpha=0.5) + ab.pl.gradient_quiver(mdata, gene, bin_size_um=200)
    | ab.pl.points(mdata, color=gene2, alpha=0.5) + ab.pl.gradient_quiver(mdata, gene2, bin_size_um=200)
)
bottom = (
    (ab.pl.points(mdata, color=gene, alpha=0.3) + ab.pl.gradient_streamlines(mdata, gene))
    | ab.pl.divergence_plot(mdata, gene)
    | ab.pl.flux_plot(mdata, gene)
)
(top / bottom).show()

## Where to go next

This notebook computed directionality from already-available gene expression. A related question is whether those directions correlate with cell-type boundaries (covered in `03_neighborhood_analysis.ipynb`) or with cell-type-specific gene programs discovered via SOM (end of `06_trajectory_and_gene_modules.ipynb`). The `compartment_flux` results can be read alongside the `nhood_enrichment` and `co_occurrence` results from notebook 03 to build a fuller picture: not just which cell types sit next to each other, but in which direction their marker programs point.